# 02 — Generator selection

Operational specification: [`docs/PROTOCOL.md`, Section 2](../../docs/PROTOCOL.md#2-generator-benchmark).

This results-only notebook selects one fine-tuned and one from-scratch generator from the benchmark
summary. It loads no encoder, regenerates no image, and never accesses test data. Eligibility uses the
technical/scientific safety gates only (minimum image count, exact-duplicate and train-memorization
rates, corruption, metric completeness, test isolation, and registry role); perceptual-hash rate and
RAD-DINO coverage are descriptive ranking metrics. Each manual choice is validated against the registry
and those gates, and the selection is written as a simple, authoritative record of the G02/G07 decision.

## 1. Load the benchmark summary and registry

The cell resolves the canonical `generator_summary.csv` and loads the registry and protocol. The displayed source
path and row count make the chosen summary explicit before ranking.

In [ ]:
from pathlib import Path
import json
import sys
import csv
ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'configs').is_dir())
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / 'notebooks/utility'))

# Execution contract: offline and non-mutating unless deliberately opted into.
# See notebooks/utility/review_mode.py for the flags and the environment overrides.
import review_mode
ALLOW_NETWORK_ACCESS = False
INSTALL_DEPENDENCIES = False
ALLOW_PROCESSED_DOWNLOAD = False
review_mode.activate(
    allow_network=ALLOW_NETWORK_ACCESS,
    allow_dependency_install=INSTALL_DEPENDENCIES,
    allow_processed_download=ALLOW_PROCESSED_DOWNLOAD,
)
from notebooks.utility.generator_benchmark import load_protocol, load_registry, rank_generator_family, save_selected_generators
protocol = load_protocol(ROOT)
load_registry(ROOT)
canonical_metrics_path = ROOT / protocol['outputs']['metrics']
metrics_path = canonical_metrics_path
benchmark_rows = list(csv.DictReader(metrics_path.open())) if metrics_path.is_file() else []
paired_path = ROOT / protocol['outputs']['paired_differences']
paired_rows = list(csv.DictReader(paired_path.open())) if paired_path.is_file() else []
{'benchmark_summary_source': str(metrics_path.relative_to(ROOT)),
 'n_benchmark_rows': len(benchmark_rows)} if benchmark_rows else 'Not yet evaluated'


## 2. Rank each family under the technical safety gates

Each family is ranked by the protocol-defined RAD-DINO-KID-primary hierarchy under the technical safety
gates only. Perceptual-hash-only similarity and RAD-DINO coverage are descriptive metrics rather than
binary gates; the remaining eligibility inputs are minimum image count, exact/confirmed duplication,
train memorization, corruption, metric completeness, test isolation, and the registry role. The auxiliary
candidate audit is not used as an eligibility gate.

In [ ]:
filtered_rows = [row for row in benchmark_rows if row.get('condition') == 'FILTERED']
gates = protocol['eligibility_gates']
finetuned_ranking = rank_generator_family(filtered_rows, 'finetuned', gates) if filtered_rows else []
fromscratch_ranking = rank_generator_family(filtered_rows, 'from_scratch', gates) if filtered_rows else []
outcome = {'eligible': sum(bool(row['eligible']) for row in finetuned_ranking + fromscratch_ranking),
           'exclusions': [(row['generator_id'], row['exclusion_reasons']) for row in finetuned_ranking + fromscratch_ranking],
           'finetuned_rank': [(row['generator_id'], row['family_rank']) for row in finetuned_ranking],
           'from_scratch_rank': [(row['generator_id'], row['family_rank']) for row in fromscratch_ranking]}
outcome


## 3. Display the descriptive evidence and the KID-primary hierarchy

The tables expose fidelity, coverage, stability, duplication, memorization, and efficiency fields for
the candidates present in the selected summary. Within each family, ordering is RAD-DINO-KID-primary
with the registered deterministic tie-breaks; non-selectable rows remain visible for transparency.

In [ ]:
display_columns = ['generator_id', 'family_rank', 'raddino_kid', 'raddino_kid_stability_low', 'raddino_kid_stability_high',
                   'raddino_coverage', 'raddino_precision', 'raddino_fid', 'inception_kid', 'raddino_kid_std',
                   'perceptual_hash_duplicate_rate',  # descriptive only, not a gate
                   'train_memorization_rate', 'synthetic_exact_duplicate_rate',
                   'generation_seconds_per_image', 'efficiency_status']
[[{column: row.get(column) for column in display_columns} for row in ranking] for ranking in (finetuned_ranking, fromscratch_ranking)]


## 4. Review paired differences and declare the two manual choices

Paired repeated-subsampling differences describe how candidate KID estimates move under the shared
sampling plan. The proposed top-ranked candidates are displayed beside the two explicit manual
constants. Final validation checks family membership, registry selection eligibility, metric
completeness, the 1,361-image requirement, test isolation, and the technical safety gates; it does not
require the manual choice to equal the proposed top-ranked row.

In [ ]:
SELECTED_FINETUNED_GENERATOR = "02_sd21_filtered_100steps"
SELECTED_FROM_SCRATCH_GENERATOR = "07_ldm_sdvae_extra1361"
PROPOSED_FINETUNED_GENERATOR = next((row['generator_id'] for row in finetuned_ranking if row['eligible']), None)
PROPOSED_FROM_SCRATCH_GENERATOR = next((row['generator_id'] for row in fromscratch_ranking if row['eligible']), None)
SELECTION_NOTES = ('Eligibility uses the technical/scientific safety gates only (image count, exact-duplicate and '
                   'train-memorization rates, corruption, metric completeness, test isolation, registry role); '
                   'perceptual-hash rate and RAD-DINO coverage are descriptive ranking metrics. The protocol-defined '
                   'RAD-DINO KID-primary hierarchy selects G02 (fine-tuned) and G07 (from-scratch).')
{'selected': (SELECTED_FINETUNED_GENERATOR, SELECTED_FROM_SCRATCH_GENERATOR),
 'proposed_top_rank': (PROPOSED_FINETUNED_GENERATOR, PROPOSED_FROM_SCRATCH_GENERATOR),
 'paired_generator_differences': paired_rows}


## 5. Validate and persist the selection

With `SAVE_SELECTION=True`, the two validated choices are written atomically to the canonical selection
file (`configs/selected_generators.json`) — the two generator IDs, their family, descriptive rank and
primary-metric value, and `test_access = false`. The file is deliberately small; classifier synthetic conditions read the selected generator's
canonical FILTERED pool directly. Setting the flag to `False` provides a read-only review.

In [ ]:
SAVE_SELECTION = False  # explicit opt-in; review mode never rewrites the selection
if SAVE_SELECTION and benchmark_rows:
    output = save_selected_generators(ROOT, SELECTED_FINETUNED_GENERATOR, SELECTED_FROM_SCRATCH_GENERATOR,
                                      benchmark_rows, notes=SELECTION_NOTES)
    print('Saved selection to', output)
    print(json.dumps(json.loads(Path(output).read_text()), indent=1))
else:
    print('Selection not saved. Requires benchmark results.')
